In [ ]:
!pip install --upgrade torch torchvision torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.2/821.2 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20

In [ ]:
# just checking if its installed
import torch
import torchvision
import torchaudio

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Torch version: 2.7.1+cu126
CUDA available: False


# Using data loader of Daphne

In [ ]:
import numpy as np
import pandas as pd
import os
import torch
from collections import Counter
from torch.utils.data import Dataset, DataLoader

In [ ]:
# --- STEP 1: Mount Google Drive ---
# This is the initial step to make your Google Drive files accessible within Google Colab.
# When you run this cell, a pop-up window will appear asking you to authorize Colab
# to access your Google Drive. You'll need to select your Google account and grant permissions.
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
# --- Configuration ---
# This section defines the paths to your data files within your Google Drive.
# It's crucial to set these paths correctly so your code can find the data.

# `google_drive_project_path` points to the folder in your Google Drive
# where your main project files (including the 'data' subfolder) are located.
# Based on your previous information, your data is in 'My Drive/DnARnAProject/data'.
# So, the 'DnARnAProject' folder is the direct parent of your 'data' folder.
google_drive_project_path = '/content/gdrive/MyDrive/DnARnAProject/'


# `data_dir` constructs the full path to your 'data' folder by joining the project path
# with the 'data' folder name. This is where data.npz, regions.parquet, etc., are stored.
data_dir = os.path.join(google_drive_project_path, 'data/')



In [ ]:
print(f"Attempting to access data in: {data_dir}")
# --- Verify the data directory exists ---
# This block checks if the specified data directory actually exists in your mounted Google Drive.
# It's a critical debugging step to ensure your path is correct before attempting to load data.
if not os.path.isdir(data_dir):
    print(f"Error: The directory '{data_dir}' does not exist.")
    print("Please check your Google Drive path and folder names for typos.")
    print("Listing contents of your project folder for debugging:")
    try:
        # This helps in debugging by showing what's actually inside the assumed project path.
        # It can help you spot if 'DnARnAProject' is misspelled or located elsewhere.
        print(os.listdir(google_drive_project_path))
    except FileNotFoundError:
        # Handles the case where even the parent project folder isn't found.
        print(f"Project folder '{google_drive_project_path}' not found either. Check the full path.")
    # Note: We don't use exit() here in a Colab notebook to avoid stopping the entire runtime.
    # However, subsequent cells relying on `data_dir` will likely fail if the path is wrong.


Attempting to access data in: /content/gdrive/MyDrive/DnARnAProject/data/


## Load data.npz

In [ ]:
# This file contains the concatenated sequence and expression data for all chromosomes.
print("--- Inspecting data.npz ---")
try:
    data_npz_path = os.path.join(data_dir, 'data.npz')
    # np.load() is used to load .npz files, which are zipped NumPy arrays.
    data_npz = np.load(data_npz_path)

    print(f"Keys in data.npz: {list(data_npz.keys())}") # Shows what arrays are stored inside the .npz file.

    # Check for 'sequence' data, its shape, data type, and a sample of values.
    if 'sequence' in data_npz:
        seq_array = data_npz['sequence']
        print(f"Sequence array shape: {seq_array.shape}") # (num_bases_total,) indicating a 1D array of base encodings.
        print(f"Sequence array dtype: {seq_array.dtype}") # Typically uint8 for integer encodings (A=0, C=1, G=2, T=3, N=4).
        print(f"First 100 sequence values: {seq_array[:100]}")
        # Sample unique values to confirm the encoding scheme (e.g., 0, 1, 2, 3, 4).
        unique_seq_values = np.unique(seq_array[:10000])
        print(f"Unique values in sequence (sample): {unique_seq_values}")
        counts = dict(Counter(seq_array))
        print("Base counts in entire sequence array:")
        print(counts)
        print("Unique values in first 1000 bases:", np.unique(seq_array[:1000]))
        print("Unique values in first 10000 bases:", np.unique(seq_array[:10000]))


    # Check for 'expression_plus' data (forward strand expression).
    if 'expressed_plus' in data_npz:
        expr_plus_array = data_npz['expressed_plus']
        print(f"Expression_plus array shape: {expr_plus_array.shape}") # (num_bases_total,) similar to sequence.
        print(f"Expression_plus array dtype: {expr_plus_array.dtype}") # Typically uint8 (expressed=1, unexpressed=0).
        print(f"First 10 expression_plus values: {expr_plus_array[:10]}")
        unique_expr_values = np.unique(expr_plus_array[:10000])
        print(f"Unique values in expression_plus (sample): {unique_expr_values}")

    # Check for 'expression_minus' data (backward strand expression).
    if 'expressed_minus' in data_npz:
        expr_minus_array = data_npz['expressed_minus']
        print(f"Expression_minus array shape: {expr_minus_array.shape}")
        print(f"Expression_minus array dtype: {expr_minus_array.dtype}")
        print(f"First 10 expression_minus values: {expr_minus_array[:10]}")
        unique_expr_values = np.unique(expr_minus_array[:10000])
        print(f"Unique values in expression_minus (sample): {unique_expr_values}")

except FileNotFoundError:
    print(f"Error: {data_npz_path} not found. This typically means the path is incorrect after mounting Drive.")
    print("Double-check the folder names in your Google Drive for typos.")
except Exception as e:
    print(f"An error occurred while loading data.npz: {e}")


--- Inspecting data.npz ---
Keys in data.npz: ['sequence', 'expressed_plus', 'expressed_minus']
Sequence array shape: (12157105,)
Sequence array dtype: uint8
First 100 sequence values: [1 1 0 1 0 1 1 0 1 0 1 1 1 0 1 0 1 0 1 1 1 0 1 0 1 0 1 1 0 1 0 1 1 0 1 0 1
 0 1 1 0 1 0 1 1 0 1 0 1 1 1 0 1 0 1 0 1 0 1 0 1 0 3 1 1 3 0 0 1 0 1 3 0 1
 1 1 3 0 0 1 0 1 0 2 1 1 1 3 0 0 3 1 3 0 0 1 1 1 3 2]
Unique values in sequence (sample): [0 1 2 3]
Base counts in entire sequence array:
{np.uint8(1): 2320576, np.uint8(0): 3766349, np.uint8(3): 3753080, np.uint8(2): 2317100}
Unique values in first 1000 bases: [0 1 2 3]
Unique values in first 10000 bases: [0 1 2 3]
Expression_plus array shape: (12157105,)
Expression_plus array dtype: uint8
First 10 expression_plus values: [0 0 0 0 0 0 0 0 0 0]
Unique values in expression_plus (sample): [0 1]
Expression_minus array shape: (12157105,)
Expression_minus array dtype: uint8
First 10 expression_minus values: [0 0 0 0 0 0 0 0 0 0]
Unique values in expression_minus

Can the plus and minus strand be expressed at the same time?

In [ ]:
expressed_plus = data_npz['expressed_plus']
expressed_minus = data_npz['expressed_minus']

# Check for co-expression
both_expressed = (expressed_plus == 1) & (expressed_minus == 1)
count_both = np.sum(both_expressed)

# Total positions
total = expressed_plus.shape[0]

print(f"Positions where both + and - strand are expressed: {count_both}")
print(f"Percentage of total: {100 * count_both / total:.6f}%")

Positions where both + and - strand are expressed: 267945
Percentage of total: 2.204020%


## Load regions.parquet

In [ ]:
# This section loads and inspects 'regions.parquet'.
# This file contains suggested training regions with their offsets and window sizes.
print("\n--- Inspecting regions.parquet ---")
try:
    regions_parquet_path = os.path.join(data_dir, 'regions.parquet')
    # pd.read_parquet() is used to load data from Parquet files into a Pandas DataFrame.
    regions_df = pd.read_parquet(regions_parquet_path)

    print(f"Regions DataFrame shape: {regions_df.shape}") # Displays the number of rows (regions) and columns.
    print("Regions DataFrame head:")
    print(regions_df.head()) # Shows the first few rows of the DataFrame, providing a quick look at the data.
    print("\nRegions DataFrame info:")
    regions_df.info() # Provides a summary of the DataFrame, including data types and non-null values for each column.

    print(f"\nUnique strands: {regions_df['strand'].unique()}") # Checks for unique values in the 'strand' column (e.g., '+' or '-').

except FileNotFoundError:
    print(f"Error: {regions_parquet_path} not found. This typically means the path is incorrect after mounting Drive.")
except Exception as e:
    print(f"An error occurred while loading regions.parquet: {e}")


--- Inspecting regions.parquet ---
Regions DataFrame shape: (15705825, 6)
Regions DataFrame head:
  contig strand  start  offset  window_size  num_expressed
0   chrI      +   7655    7655         2048            205
1   chrI      +   7656    7656         2048            206
2   chrI      +   7657    7657         2048            207
3   chrI      +   7658    7658         2048            208
4   chrI      +   7659    7659         2048            209

Regions DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15705825 entries, 0 to 15705824
Data columns (total 6 columns):
 #   Column         Dtype 
---  ------         ----- 
 0   contig         object
 1   strand         object
 2   start          int64 
 3   offset         int64 
 4   window_size    int64 
 5   num_expressed  int64 
dtypes: int64(4), object(2)
memory usage: 719.0+ MB

Unique strands: ['+' '-']


In [ ]:
unique_contigs = regions_df['contig'].unique()
print("Unique contigs (chromosomes or scaffolds):", unique_contigs)
print("Number of unique contigs:", len(unique_contigs))


Unique contigs (chromosomes or scaffolds): ['chrI' 'chrII' 'chrIII' 'chrIV' 'chrIX' 'chrV' 'chrVI' 'chrVII' 'chrVIII'
 'chrX' 'chrXI' 'chrXII' 'chrXIII' 'chrXIV' 'chrXV' 'chrXVI' 'chrM']
Number of unique contigs: 17


## Load ensembl_annotation.gff3

In [ ]:
# This is an annotation file from Ensembl and is optional for the DataLoader itself,
# but can be useful for understanding the problem.
print("\n--- Inspecting ensembl_annotation.gff3 (Optional) ---")
try:
    gff3_path = os.path.join(data_dir, 'ensembl_annotation.gff3')
    # This block opens the GFF3 file and prints its first 10 lines to give a glimpse of its format.
    with open(gff3_path, "r") as handle:
        gff_lines = [next(handle) for _ in range(10)]
        print("First 10 lines of GFF3 file:")
        for line in gff_lines:
            print(line.strip()) # .strip() removes leading/trailing whitespace including newlines.

except FileNotFoundError:
    print(f"Error: {gff3_path} not found. This file is optional for the DataLoader.")
except Exception as e:
    print(f"An error occurred while inspecting ensembl_annotation.gff3: {e}")



--- Inspecting ensembl_annotation.gff3 (Optional) ---
First 10 lines of GFF3 file:
##gff-version 3
##sequence-region   I 1 230218
##sequence-region   II 1 813184
##sequence-region   III 1 316620
##sequence-region   IV 1 1531933
##sequence-region   IX 1 439888
##sequence-region   Mito 1 85779
##sequence-region   V 1 576874
##sequence-region   VI 1 270161
##sequence-region   VII 1 1090940


## Creating a Data Loader class

In [ ]:
# --- Custom PyTorch DataLoader Class (same as before) ---
# This is your core data loading class for PyTorch, inheriting from torch.utils.data.Dataset.
# It defines how individual data samples (sequence segments and expression labels) are loaded.
class GenomeExpressionDataset(Dataset):
    """
    Custom Dataset for loading DNA sequence and expression data for genomic regions.
    It loads data from pre-processed .npz and .parquet files.
    """
    def __init__(self, data_dir):
        """
        Initializes the dataset by loading the full sequence and expression arrays
        and the DataFrame of genomic regions.

        Args:
            data_dir (str): The path to the directory containing 'data.npz' and 'regions.parquet'.
        """
        self.data_dir = data_dir

        # Construct full paths to data files
        self.data_npz_path = os.path.join(data_dir, 'data.npz')
        self.regions_parquet_path = os.path.join(data_dir, 'regions.parquet')

        # Load data.npz containing sequence and expression arrays
        try:
            self.data_npz = np.load(self.data_npz_path)
            self.sequence_data = self.data_npz['sequence'] # Array of encoded DNA bases (0-4)
            # IMPORTANT: Corrected key to 'expressed_plus' based on your data.npz keys
            self.expression_plus_data = self.data_npz['expressed_plus'] # Array of expression labels for forward strand (0 or 1)
            # IMPORTANT: Corrected key to 'expressed_minus' based on your data.npz keys
            self.expression_minus_data = self.data_npz['expressed_minus'] # Array of expression labels for reverse strand (0 or 1)
        except Exception as e:
            raise RuntimeError(f"Could not load data from {self.data_npz_path}. Make sure the file exists and is not corrupted: {e}")

        # Load regions.parquet containing metadata for each genomic region
        try:
            self.regions_df = pd.read_parquet(self.regions_parquet_path)
        except Exception as e:
            raise RuntimeError(f"Could not load regions from {self.regions_parquet_path}. Make sure the file exists and is not corrupted: {e}")

        # Number of unique nucleotides/channels for one-hot encoding (A, C, G, T, N)
        self.num_nucleotides = 5

    def __len__(self):
        """
        Returns the total number of samples (genomic regions) in the dataset.
        This is determined by the number of rows in the regions DataFrame.
        """
        return len(self.regions_df)

    def _one_hot_encode(self, sequence_segment):
        """
        Converts a sequence segment (array of integer encodings) into a one-hot encoded tensor.
        Example: [0, 1, 2] (A, C, G) -> [[1,0,0,0,0], [0,1,0,0,0], [0,0,1,0,0]]
        """
        one_hot_tensor = torch.zeros(len(sequence_segment), self.num_nucleotides, dtype=torch.float32)
        one_hot_tensor.scatter_(1, torch.tensor(sequence_segment).unsqueeze(1).long(), 1)
        return one_hot_tensor

    def __getitem__(self, idx):
        """
        Retrieves a single data sample (sequence, expression label, and metadata) by its index.

        Args:
            idx (int or torch.Tensor): The index of the region to retrieve.

        Returns:
            tuple: A tuple containing:
                - encoded_sequence (torch.Tensor): The one-hot encoded DNA sequence segment.
                - expression_label (torch.Tensor): The expression label (0 or 1) for the region.
                - region_info (dict): A dictionary containing metadata for the region (e.g., contig, strand, offset).
        """
        if torch.is_tensor(idx):
            idx = idx.tolist()

        region_info = self.regions_df.iloc[idx]

        offset = region_info['offset'] #
        window_size = region_info['window_size'] #
        strand = region_info['strand']

        sequence_segment = self.sequence_data[offset : offset + window_size]
        encoded_sequence = self._one_hot_encode(sequence_segment)

        #if strand == '+':
        #    expression_label = self.expression_plus_data[offset]
        #else:
        #    expression_label = self.expression_minus_data[offset]

        #expression_label = torch.tensor(expression_label, dtype=torch.long)

        # CHANGE: had to make changes to the above code because for training i want prediction value for each nucleotide, not for each window
        if strand == '+':
            expression_label = self.expression_plus_data[offset : offset + window_size]
        else:
            expression_label = self.expression_minus_data[offset : offset + window_size]

        expression_label = torch.tensor(expression_label, dtype=torch.long)


        return encoded_sequence, expression_label, region_info.to_dict()

### checking if it's initialized correctly--> DUE TO CHANGES TO THE LOADER THIS DOESN'T WORK, ITS EXPECTED

In [ ]:
# --- Instantiating and trying out the class ---
print("\n--- Instantiating and Testing GenomeExpressionDataset ---")
try:
    # 1. Instantiate the dataset:
    # Pass the 'data_dir' variable which points to your 'data' folder in Google Drive.
    my_dataset = GenomeExpressionDataset(data_dir=data_dir)
    print(f"Successfully instantiated GenomeExpressionDataset.")
    print(f"Total number of samples (regions) in the dataset: {len(my_dataset)}")

    # 2. Access a single sample using __getitem__
    # You can access individual samples by their index, like a list.
    print("\n--- Accessing individual samples ---")
    sample_index_0 = 0 # Get the first sample
    sequence_0, label_0, metadata_0 = my_dataset[sample_index_0]
    print(f"Sample at index {sample_index_0}:")
    print(f"  Sequence shape: {sequence_0.shape} (One-hot encoded)")
    print(f"  Label: {label_0.item()}") # .item() gets the scalar value from a 0-dim tensor
    print(f"  Metadata: {metadata_0}")

    # Get another sample, e.g., at a different index
    sample_index_500 = 500 # Get the 501st sample
    sequence_500, label_500, metadata_500 = my_dataset[sample_index_500]
    print(f"\nSample at index {sample_index_500}:")
    print(f"  Sequence shape: {sequence_500.shape} (One-hot encoded)")
    print(f"  Label: {label_500.item()}")
    print(f"  Metadata: {metadata_500}")

    # 3. Use it with PyTorch's DataLoader
    # The DataLoader is what typically iterates over your dataset in batches during training.
    print("\n--- Using DataLoader to get batches ---")
    batch_size = 8 # Define your desired batch size
    # num_workers=0 is usually recommended for debugging in Colab to avoid multiprocessing issues.
    # Set to >0 for faster data loading in production.
    data_loader = DataLoader(my_dataset, batch_size=batch_size, shuffle=True, num_workers=0)

    # Iterate through a few batches
    for batch_idx, (sequences, labels, metadata) in enumerate(data_loader):
        print(f"\n--- Batch {batch_idx + 1} ---")
        print(f"  Batch of sequences shape: {sequences.shape}") # (batch_size, window_size, num_nucleotides)
        print(f"  Batch of labels shape: {labels.shape}")       # (batch_size,)
        print(f"  Labels in this batch: {labels.numpy()}") # .numpy() to see the values easily
        print(f"  Metadata for first sample in batch:")
        print(f"    Contig: {metadata['contig'][0]}, Strand: {metadata['strand'][0]}, Offset: {metadata['offset'][0]}")

        if batch_idx >= 1: # Print only the first 2 batches for brevity
            break

except RuntimeError as e:
    print(f"\nError during dataset instantiation or usage: {e}")
    print("Please ensure your Google Drive path is correct and data files are accessible.")
except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")


--- Instantiating and Testing GenomeExpressionDataset ---
Successfully instantiated GenomeExpressionDataset.
Total number of samples (regions) in the dataset: 15705825

--- Accessing individual samples ---
Sample at index 0:
  Sequence shape: torch.Size([2048, 5]) (One-hot encoded)

Error during dataset instantiation or usage: a Tensor with 2048 elements cannot be converted to Scalar
Please ensure your Google Drive path is correct and data files are accessible.


# First CNN --> this was a wrong trial since we understood the input format wrong

In [ ]:
dataset = GenomeExpressionDataset(data_dir)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

In [ ]:
for seq, expr, _ in dataloader:
    print(seq.shape)   # (batch_size, window_size, 5)
    print(expr.shape)  # (batch_size, window_size)
    break


torch.Size([16, 2048, 5])
torch.Size([16, 2048])


In [ ]:
import torch.nn as nn

class CNNExpressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(5, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(64, 1, kernel_size=1)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.transpose(1, 2)  # (B, 5, L)
        x = self.cnn(x)        # (B, 1, L)
        x = self.sigmoid(x)
        return x.squeeze(1)    # (B, L)


In [ ]:
# Automatically use GPU if available, otherwise fallback to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


### !!!! training unsuccessful --> data too big

In [ ]:
import torch.optim as optim

model = CNNExpressionModel().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

for epoch in range(3):  # Just 3 epochs is enough for first demo
    model.train()
    total_loss = 0
    for seq, expr, _ in dataloader:
        seq, expr = seq.to(device), expr.float().to(device)

        optimizer.zero_grad()
        output = model(seq)
        loss = criterion(output, expr)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


Epoch 1, Loss: 666959.3591


### TRAIN with 10% of the data initially

In [ ]:
total_size = len(dataset)
subset_size = int(0.05 * total_size)  # 5%

np.random.seed(42)  # For reproducibility
subset_indices = np.random.choice(total_size, size=subset_size, replace=False)
subset_indices = sorted(subset_indices)

# Step 3: Create subset
subset_dataset = Subset(dataset, subset_indices)

# Step 4: Use it in DataLoader
dataloader = DataLoader(subset_dataset, batch_size=16, shuffle=True)

# Train CNN that takes sequence and expression data and masks while training

### creating a smaller dataset for initial testing --> 0.2%

In [ ]:
from torch.utils.data import Subset

In [ ]:
dataset = GenomeExpressionDataset(data_dir)  # your full dataset
total = len(dataset)
fraction = 0.002  # 0.2%
n_samples = int(total * fraction)

# Randomly sample indices
np.random.seed(42)
subset_indices = np.random.choice(len(dataset), size=n_samples, replace=False)
subset = Subset(dataset, subset_indices)

# Use it in the DataLoader
subset_loader = DataLoader(subset, batch_size=16, shuffle=True)


In [ ]:
# just for visualizing --> not the one used for training since smaller batch size
subset_loader_vis = DataLoader(subset, batch_size=1, shuffle=False)

# Show first few samples
print(f"Subset contains {len(subset)} regions\n")

for i, (dna, expr, meta) in enumerate(subset_loader_vis):
    print(f"Sample {i+1}")
    print(f"Sequence shape: {dna.shape}")  # (1, window_size, 5)
    print(f"Expression shape: {expr.shape}")  # (1, window_size)
    print(f"Metadata: {meta}")

    print("First 20 DNA bases (one-hot index):")
    print(torch.argmax(dna[0, :20], dim=-1).tolist())  # Converts one-hot back to index
    print("First 20 expression labels:")
    print(expr[0, :20].tolist())

    print("-----------")

    if i >= 2:  # Just show 3 examples
        break

Subset contains 31411 regions

Sample 1
Sequence shape: torch.Size([1, 2048, 5])
Expression shape: torch.Size([1, 2048])
Metadata: {'contig': ['chrXVI'], 'strand': ['-'], 'start': tensor([335079]), 'offset': tensor([11458339]), 'window_size': tensor([2048]), 'num_expressed': tensor([1289])}
First 20 DNA bases (one-hot index):
[0, 0, 3, 2, 2, 0, 2, 2, 2, 2, 0, 1, 0, 0, 3, 2, 3, 1, 2, 2]
First 20 expression labels:
[0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
-----------
Sample 2
Sequence shape: torch.Size([1, 2048, 5])
Expression shape: torch.Size([1, 2048])
Metadata: {'contig': ['chrX'], 'strand': ['-'], 'start': tensor([310099]), 'offset': tensor([6142560]), 'window_size': tensor([2048]), 'num_expressed': tensor([1632])}
First 20 DNA bases (one-hot index):
[0, 2, 3, 1, 0, 2, 0, 2, 2, 2, 0, 3, 1, 3, 3, 2, 2, 0, 0, 3]
First 20 expression labels:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
-----------
Sample 3
Sequence shape: torch.Size([1, 2048, 5])
Expr

### BPnet CNN model

In [ ]:
class BPNet(torch.nn.Module):
    def __init__(
        self,
        in_channels: int = 5 + 2 + 1,
        n_layers: int = 7,
        n_filters: int = 64,
        first_layer_kernel_size: int = 7,
        dilation: bool = True,
        kernel_size: int = 3,
        final_kernel_size: int = 30,
        out_channels: int = 1,
    ):
        super().__init__()
        self.n_layers = n_layers

        self.iconv = torch.nn.Conv1d(
            in_channels, n_filters, kernel_size=first_layer_kernel_size, padding="same"
        )
        self.irelu = torch.nn.ReLU()
        self.rconvs = torch.nn.ModuleList(
            [
                torch.nn.Conv1d(
                    n_filters,
                    n_filters,
                    kernel_size=kernel_size,
                    dilation=2**i if dilation else 1,
                    padding="same",
                )
                for i in range(1, self.n_layers + 1)
            ]
        )
        self.rrelus = torch.nn.ModuleList([torch.nn.ReLU() for i in range(1, self.n_layers + 1)])
        self.out_conv = torch.nn.Conv1d(
            n_filters, out_channels, kernel_size=final_kernel_size, padding="same"
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.transpose(-2, -1)
        x = self.iconv(x)
        x = self.irelu(x)
        for i in range(self.n_layers):
            x = x + self.rrelus[i](self.rconvs[i](x))
        x = self.out_conv(x)
        return x.transpose(-2, -1)


### Training with 0.2% of the dataset

In [ ]:
import torch.nn.functional as F

In [ ]:
# Assume:
# - `subset_loader` yields: (one_hot_dna, true_expr, metadata)
# - Model: BPNet (output shape: B x L x 1 or B x L)
# - Input: concat of DNA + masked RNA + mask indicator → shape (B, L, 8)
# - Loss: only on masked positions
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = BPNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.BCEWithLogitsLoss()

num_epochs = 3
print_every = 10

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    print(f"\n🔄 Starting epoch {epoch+1}/{num_epochs}")

    for batch_idx, (dna, expr, _) in enumerate(subset_loader):
        dna = dna.to(device)               # (B, L, 5)
        expr = expr.to(device).float()     # (B, L)

        B, L, _ = dna.shape

        # Create mask: mask = False for masked positions
        mask = torch.ones((B, L), dtype=torch.bool, device=device)
        mask[:, :L // 2] = False

        # Expression input (B, L, 2)
        expr_plus = expr.unsqueeze(-1)
        expr_minus = torch.zeros_like(expr_plus)
        expr_input = torch.cat([expr_plus, expr_minus], dim=-1)

        # Apply masking value (0.5)
        expr_input[~mask, :] = 0.5

        # Add mask channel
        mask_channel = mask.float().unsqueeze(-1)  # (B, L, 1)

        # Final input: (B, L, 8)
        model_input = torch.cat([dna, expr_input, mask_channel], dim=-1)

        # Forward pass
        output = model(model_input).squeeze(-1)  # (B, L)

        # Loss on masked positions
        loss = criterion(output[~mask], expr[~mask])

        # Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Print progress
        if (batch_idx + 1) % print_every == 0 or (batch_idx == 0):
            print(f"  📦 Batch {batch_idx+1:3d} | Loss: {loss.item():.4f}")

    print(f"✅ Finished epoch {epoch+1} | Total Loss: {total_loss:.4f}")

Using device: cpu

🔄 Starting epoch 1/3
  📦 Batch   1 | Loss: 0.7238
  📦 Batch  10 | Loss: 0.6814
  📦 Batch  20 | Loss: 0.6884
  📦 Batch  30 | Loss: 0.7117
  📦 Batch  40 | Loss: 0.6380
  📦 Batch  50 | Loss: 0.6601
  📦 Batch  60 | Loss: 0.6320
  📦 Batch  70 | Loss: 0.6622
  📦 Batch  80 | Loss: 0.6066
  📦 Batch  90 | Loss: 0.6341
  📦 Batch 100 | Loss: 0.6576
  📦 Batch 110 | Loss: 0.6649
  📦 Batch 120 | Loss: 0.6545
  📦 Batch 130 | Loss: 0.6586
  📦 Batch 140 | Loss: 0.6339
  📦 Batch 150 | Loss: 0.6486
  📦 Batch 160 | Loss: 0.6214
  📦 Batch 170 | Loss: 0.6410
  📦 Batch 180 | Loss: 0.6658
  📦 Batch 190 | Loss: 0.6357
  📦 Batch 200 | Loss: 0.6640
  📦 Batch 210 | Loss: 0.6723
  📦 Batch 220 | Loss: 0.6100
  📦 Batch 230 | Loss: 0.5986
  📦 Batch 240 | Loss: 0.6245
  📦 Batch 250 | Loss: 0.6648
  📦 Batch 260 | Loss: 0.6239
  📦 Batch 270 | Loss: 0.5927
  📦 Batch 280 | Loss: 0.5775
  📦 Batch 290 | Loss: 0.6118
  📦 Batch 300 | Loss: 0.5863
  📦 Batch 310 | Loss: 0.6619
  📦 Batch 320 | Loss: 0.6086
  📦

#### Saving the trained model

In [ ]:
torch.save(model.state_dict(), "bpnet_model.pt")

In [ ]:
torch.save(model.state_dict(), "/content/gdrive/MyDrive/bpnet_model.pt")